In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [3]:
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [4]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings(model="text-embedding-3-large")

In [5]:
embeddings.embed_query("hello world")

[-0.0027637481689453125,
 -0.022674560546875,
 0.00510406494140625,
 0.0269317626953125,
 0.01532745361328125,
 0.00980377197265625,
 -0.005542755126953125,
 0.08544921875,
 0.01155853271484375,
 0.047698974609375,
 -0.00472259521484375,
 0.0009427070617675781,
 -0.01105499267578125,
 -0.0177764892578125,
 -0.0097198486328125,
 0.03912353515625,
 -0.0128631591796875,
 -0.005512237548828125,
 -0.01309967041015625,
 -0.01544189453125,
 0.022308349609375,
 -0.0054779052734375,
 -0.0146484375,
 0.055145263671875,
 0.001071929931640625,
 0.03155517578125,
 -0.0070648193359375,
 0.0031261444091796875,
 -0.050750732421875,
 -0.043365478515625,
 0.0167083740234375,
 0.01245880126953125,
 -0.00177001953125,
 0.01464080810546875,
 0.03326416015625,
 -0.0015001296997070312,
 0.0296783447265625,
 0.00984954833984375,
 0.0098114013671875,
 0.00867462158203125,
 0.040008544921875,
 0.0265960693359375,
 -0.0227508544921875,
 0.0171356201171875,
 -0.00525665283203125,
 0.007480621337890625,
 0.0027923

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
documents=["Washington is the capital of USA",
           "Donald Trump is a president of USA",
           "Narendra Modi is a prime minister of India."]

In [8]:
my_query  = "Who is a prime minister of India?"

In [9]:
embedded_docs=embeddings.embed_documents(documents)
embedded_query=embeddings.embed_query(my_query)

In [10]:
cosine_similarity([embedded_query], embedded_docs)

array([[0.10408052, 0.30370604, 0.72317569]])

In [11]:
from sklearn.metrics.pairwise import euclidean_distances

In [12]:
euclidean_distances([embedded_query], embedded_docs)

array([[1.33811952, 1.18006978, 0.74397391]])

In [13]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

/var/folders/jm/7clyqzg57313wr8rz15hptym0000gn/T/ipykernel_16906/2965923628.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.docstore.in_memory import InMemoryDocstore


In [14]:
myemd=embeddings.embed_query("hello")
len(myemd)

3072

In [15]:
index=faiss.IndexFlatL2(3072)

In [16]:
# This is my vector  store
#  inside this vectorstore the data  is being stor insise the index
#  As of now it is inmomory but we can also store it in disk as well

vector_store=FAISS(embedding_function= embeddings,
      index=index,
      docstore=InMemoryDocstore(),
      index_to_docstore_id={}
      )


In [17]:
vector_store.add_texts(
           ["Washington is the capital of USA",
           "Donald Trump is a president of USA",
           "Narendra Modi is a prime minister of India."])

['ff7f1e27-6c46-4a62-98b4-f24dc3210d78',
 '56d5e1d6-666b-4545-8b57-212c8200cf8c',
 '44fbaabd-77e7-4a75-a47b-69a0e8fe0046']

In [18]:
vector_store.index_to_docstore_id

{0: 'ff7f1e27-6c46-4a62-98b4-f24dc3210d78',
 1: '56d5e1d6-666b-4545-8b57-212c8200cf8c',
 2: '44fbaabd-77e7-4a75-a47b-69a0e8fe0046'}

In [19]:
faiss_index_id =1

In [20]:
docstore_id=vector_store.index_to_docstore_id[faiss_index_id]

In [21]:
vector_store.docstore.search(docstore_id)

Document(id='56d5e1d6-666b-4545-8b57-212c8200cf8c', metadata={}, page_content='Donald Trump is a president of USA')

In [22]:
#by K  select top k result  from indexs 
vector_store.similarity_search("Who is a president of india?", k=1)

[Document(id='44fbaabd-77e7-4a75-a47b-69a0e8fe0046', metadata={}, page_content='Narendra Modi is a prime minister of India.')]

In [23]:
#by K  select top k result  from indexs 
vector_store.similarity_search("Who is a president of USA?", k=1)

[Document(id='56d5e1d6-666b-4545-8b57-212c8200cf8c', metadata={}, page_content='Donald Trump is a president of USA')]

In [24]:
#by K  select top k result  from indexs 
vector_store.similarity_search("Who is a president of USA?", k=2)

[Document(id='56d5e1d6-666b-4545-8b57-212c8200cf8c', metadata={}, page_content='Donald Trump is a president of USA'),
 Document(id='ff7f1e27-6c46-4a62-98b4-f24dc3210d78', metadata={}, page_content='Washington is the capital of USA')]